In [1]:
import pickle
import sys
import copy
import time

import cobra
import sympy
import pandas as pd
import numpy as np

from tqdm import tqdm

sys.path.insert(1, '/home/hratch/Projects/human_me/scripts/')
from utils import functions as func
from utils import parameters as params

No objective coefficients in model. Unclear what should be optimized


In [2]:
from macromolecules.RNA import RNA, mRNA
from macromolecules.protein import Protein
from macromolecules.macromolecule import Macromolecule
from macromolecules.complex import Complex

In [3]:
lp_path = '/data2/hratch/human_me/test_lp/'
with open(lp_path + 'toy_me_model.pickle', 'rb') as handle:
    me_model = pickle.load(handle)

In [4]:
# not Metabolites, not proxy (on its own), not mRNA on its own
# HGNC yes


In [5]:
lp_path = '/data2/hratch/human_me/test_lp/'

def add_metabolite(am = [], mu_val = 1e-9):
    
    with open(lp_path + 'toy_me_model.pickle', 'rb') as handle:
        me_model = pickle.load(handle)
    
    ra = []
    for m in am: #am:
        r = cobra.Reaction('TEST_' + m.id)
        r.add_metabolites({m: 1})
        ra.append(r)
    
    me_model.add_reactions(ra)
    sln, status, _ = me_model.solve_lp(mu_val = mu_val)
    return sln, status



In [6]:
lp_path = '/data2/hratch/human_me/test_lp/'

def add_metabolite(am = [], mu_val = 1e-9):
    
    with open(lp_path + 'toy_me_model.pickle', 'rb') as handle:
        me_model = pickle.load(handle)
    
    ra = []
    for m in am: #am:
        r = cobra.Reaction('TEST_' + m.id)
        r.add_metabolites({m: 1})
        ra.append(r)
    
    me_model.add_reactions(ra)
    sln, status, _ = me_model.solve_lp(mu_val = mu_val)
    return sln, status

In [73]:
me_model.reactions.get_by_id('HGNC:10420_TRANSLATION_ELONGATIONc')

Reaction identifier,HGNC:10420_TRANSLATION_ELONGATIONc
Name,
Memory address,0x07fdf6c47f7b8
Stoichiometry,(mu + 0.0693147180559945)/(275422.870333817*mu + 5508.45740667634) HGNC:10420_mrna_c + 0.0693147180559945/(275422.870333817*mu + 5508.45740667634) HGNC:10420_mrna_deg_proxy + 3.50724485351852e6*mu ... (mu + 0.0693147180559945)/(275422.870333817*mu + 5508.45740667634) + 0.0693147180559945/(275422.870333817*mu + 5508.45740667634) + 3.50724485351852e6*mu 7.01448970703703e8 + 26.927010039998095 ...
GPR,HGNC:3189 and HGNC:3214 and HGNC:3208 and HGNC:3300 and ribosome
Lower bound,0.0
Upper bound,1000.0


In [72]:
err = ['HGNC:10420', 'HGNC:10424', 'HGNC:10429', 'HGNC:10440', 'HGNC:10383', 'HGNC:10385', 'HGNC:10387', 'HGNC:10396', 
 'HGNC:10397', 'HGNC:10402', 'HGNC:10405', 'HGNC:10409', 'HGNC:10413', 'HGNC:10418', 'HGNC:3597', 'HGNC:10371', 
 'HGNC:10372', 'HGNC:10377', 'HGNC:10360', 'HGNC:10362', 'HGNC:10364', 'HGNC:21370', 'HGNC:10369', 'HGNC:10299', 
 'HGNC:10301', 'HGNC:10302', 'HGNC:10312', 'HGNC:10315', 'HGNC:10316', 'HGNC:10317', 'HGNC:10325', 'HGNC:10328', 
 'HGNC:10329', 'HGNC:10330', 'HGNC:10331', 'HGNC:10333', 'HGNC:10334', 'HGNC:10340', 'HGNC:10344', 'HGNC:10347', 
 'HGNC:10348', 'HGNC:10349', 'HGNC:17094', 'HGNC:10354', 'HGNC:12458']
err = [m_id + '_unfolded_protein_c' for m_id in err]

am = [m for m in me_model.metabolites if m.id in err[0]]
sln, status = add_metabolite(am = am)

Getting MINOS parameters...
Done in 219.439 seconds with status 0


In [70]:
test = sorted(set(flatten_list([[m.id for m in r.metabolites] for r in me_model.reactions if r.subsystem == 'Ribosome Biogenesis' and isinstance(r, func.ME_Reaction) and 'translation' in r.type])))
example = [m.id for m in me_model.metabolites if 'unfolded_protein_c' in m.id and m.id not in err and m.id in test]
example = me_model.metabolites.get_by_id(example[-9])
fail = me_model.metabolites.get_by_id(err[0])

example = [r.id for r in me_model.reactions if example.id.split('_')[0] in r.id]
fail = [r.id for r in me_model.reactions if fail.id.split('_')[0] in r.id]

In [71]:
for r_id in fail:
    r = me_model.reactions.get_by_id(r_id)
    print(r.id + ': ' + str(sln[me_model.reactions.index(r.id)]))

HGNC:10420_TRANSCRIPTION_ELONGATION: 3.8190857130157087e-22
HGNC:10420_lariats_DEGRADATIONn_0: 3.8190857130157087e-22
HGNC:10420_TRANSCRIPTION_PROCESSING: 3.8190857130157087e-22
HGNC:10420_mRNA_EXPORTtn: 3.8190857130157087e-22
HGNC:10420_DECAPPING_mRNA_DEGRADATIONc: 1.9095428427334144e-22
HGNC:10420_TRANSLATION_ELONGATIONc: 1.5175183909505576e-17
HGNC:10420_CYTOSOLIC_PROTEIN_FOLDING: 1.5175183909505576e-17
HGNC:10420_folded_protein_c_POLYUBIQUITINATIONc: -6.461895453829155e-54
HGNC:10420_folded_protein_c_DEUBIQUITINATIONc: 0.0
HGNC:10420_folded_protein_c_PROTEASOMAL_DEGRADATIONc: 0.0
HGNC:10420_IMPORTtn: 1.5175183909505576e-17


In [69]:
for r_id in fail:
    r = me_model.reactions.get_by_id(r_id)
    print(r.id + ': ' + str(sln[me_model.reactions.index(r.id)]))

HGNC:10424_TRANSCRIPTION_ELONGATION: 0.0
HGNC:10424_lariats_DEGRADATIONn_0: 0.0
HGNC:10424_TRANSCRIPTION_PROCESSING: 0.0
HGNC:10424_mRNA_EXPORTtn: 0.0
HGNC:10424_DECAPPING_mRNA_DEGRADATIONc: 0.0
HGNC:10424_TRANSLATION_ELONGATIONc: 0.0
HGNC:10424_CYTOSOLIC_PROTEIN_FOLDING: 3.955327803409348e-11
HGNC:10424_folded_protein_c_POLYUBIQUITINATIONc: 3.9553262858909565e-11
HGNC:10424_folded_protein_c_DEUBIQUITINATIONc: 0.0
HGNC:10424_folded_protein_c_PROTEASOMAL_DEGRADATIONc: 3.9553262858909565e-11
HGNC:10424_IMPORTtn: 1.5175183909505576e-17
